# TD0 Verification: Tabular vs DQN

**Goal**: Isolate whether bias is from TD0 algorithm or neural network approximation.

Three methods, same pure online TD0 update:
1. **Tabular**: dict[(m, c)] -> float (must converge by theory)
2. **DQN One-Hot**: MLP on one-hot encoded (m, c) (should work - just memorize 64 values)
3. **DQN 5-Channel**: MLP on 5-channel position encoding (might fail - needs to learn position invariance)

All use **pure online TD0**: no replay buffer, no target network.

State space: 2^N * N = 64 states for N=4 agents.

In [18]:
# Parameters (can be overridden by papermill)
PICKER_PENALTY = -1.0
SEED = 42
METHOD = "dqn_onehot"  # "tabular", "dqn_onehot", "dqn_5ch"

# Fixed settings
N_AGENTS = 4
WIDTH = 9
HEIGHT = 9
GRID_SIZE = WIDTH * HEIGHT  # 81
NUM_APPLES = 40
GAMMA = 0.99

# TD0 Training settings
TRAIN_STEPS = 1000
ALPHA = 0.1  # Learning rate for tabular
LR = 0.001    # Learning rate for DQN
LR_MIN = 1e-5
EVAL_FREQ = 10

# DQN architecture
HIDDEN_DIM = 64
NUM_LAYERS = 2


# === Isolation Flags for DQN only ===
USE_REPLAY_BUFFER = False   # Set True to isolate Buffer effect
USE_TARGET_NET = False      # Set True to isolate Target Net effect

# Hyperparameters for the isolation
BATCH_SIZE = 32
BUFFER_SIZE = 5000
TARGET_UPDATE_FREQ = 500    # Steps between target net updates

In [19]:
import numpy as np
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Dict, Tuple, List
from pathlib import Path
from itertools import product
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import os
import copy

def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

print(f"Initial RAM: {get_ram_mb():.1f} MB")

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Derived parameters
P_HIT = NUM_APPLES / GRID_SIZE
ALPHA_PROB = 1 / N_AGENTS
BETA_PROB = (N_AGENTS - 1) / N_AGENTS

# Rewards
R_PICKER = PICKER_PENALTY
R_OTHER = (1 - PICKER_PENALTY) / (N_AGENTS - 1)

print(f"METHOD = {METHOD}")
print(f"PICKER_PENALTY = {PICKER_PENALTY}")
print(f"SEED = {SEED}")
print(f"P_HIT = {P_HIT:.4f}")
print(f"R_PICKER = {R_PICKER:.4f}")
print(f"R_OTHER = {R_OTHER:.4f}")
print(f"Device = {DEVICE}")
print(f"State space size = {2**N_AGENTS * N_AGENTS}")

Initial RAM: 601.4 MB
METHOD = dqn_onehot
PICKER_PENALTY = -1.0
SEED = 42
P_HIT = 0.4938
R_PICKER = -1.0000
R_OTHER = 0.6667
Device = cpu
State space size = 64


In [20]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.memory = deque(maxlen=capacity)

    def push(self, state_t: torch.Tensor, reward_t: torch.Tensor, next_state_t: torch.Tensor):
        # Store tensors directly to avoid re-encoding overhead
        self.memory.append((state_t, reward_t, next_state_t))

    def sample(self, batch_size: int):
        transitions = random.sample(self.memory, batch_size)
        # Transpose batch of tuples to tuple of batches
        batch_state, batch_reward, batch_next_state = zip(*transitions)
        
        return (
            torch.cat(batch_state),
            torch.cat(batch_reward),
            torch.cat(batch_next_state)
        )

    def __len__(self):
        return len(self.memory)

In [21]:
@dataclass
class SimpleState:
    """Minimal state: ON/OFF status for each agent + current actor."""
    m: Tuple[bool, ...]  # tuple of bools for hashability
    c: int               # current actor index
    
    def __hash__(self):
        return hash((self.m, self.c))
    
    def __eq__(self, other):
        return self.m == other.m and self.c == other.c
    
    @staticmethod
    def from_arrays(m_array: np.ndarray, c: int) -> 'SimpleState':
        return SimpleState(m=tuple(m_array.tolist()), c=c)
    
    def to_onehot(self, n_agents: int) -> np.ndarray:
        """Convert to one-hot encoding: [m0, m1, ..., c_onehot]."""
        m_float = np.array(self.m, dtype=np.float32)
        c_onehot = np.zeros(n_agents, dtype=np.float32)
        c_onehot[self.c] = 1.0
        return np.concatenate([m_float, c_onehot])

In [22]:
s1 = SimpleState(m=(True, False, True, False), c=0)
s2 = SimpleState(m=(True, False, True, False), c=0)
print(f"s1 == s2: {s1 == s2}")
print(f"hash(s1) == hash(s2): {hash(s1) == hash(s2)}")

test_dict = {s2: 'found'}
print(f"s1 in test_dict: {s1 in test_dict}")

s1 == s2: True
hash(s1) == hash(s2): True
s1 in test_dict: True


In [23]:
def enumerate_all_states(n_agents: int) -> List[SimpleState]:
    """Enumerate all 2^n * n possible states."""
    states = []
    for m_tuple in product([False, True], repeat=n_agents):
        for c in range(n_agents):
            states.append(SimpleState(m=m_tuple, c=c))
    return states

ALL_STATES = enumerate_all_states(N_AGENTS)
print(f"Enumerated {len(ALL_STATES)} states")
print(f"RAM after enumeration: {get_ram_mb():.1f} MB")

Enumerated 64 states
RAM after enumeration: 601.4 MB


In [24]:
# Analytical value computation (from verify_mc_simple.ipynb)

def get_rho(i: int, k: int, r_picker: float, r_other: float) -> float:
    """Get reward coefficient rho^(i)_k."""
    return r_picker if k == i else r_other


def calculate_stream_values(rho: float, gamma: float, alpha: float, beta: float, p_hit: float):
    """Calculate v_RAND, v_OFF, v_ON for a given rho."""
    v_rand = (p_hit * alpha * rho) / (1 - gamma)
    coeff = (alpha * gamma) / (1 - beta * gamma)
    v_off = coeff * v_rand
    v_on = (alpha * rho) / (1 - beta * gamma) + coeff * v_rand
    return v_rand, v_off, v_on


def analytical_value(state: SimpleState, agent_i: int,
                     r_picker: float, r_other: float,
                     gamma: float, alpha: float, beta: float, p_hit: float) -> float:
    """Compute V^(i)(state) using closed-form formula."""
    m, c = state.m, state.c
    n_agents = len(m)
    
    # Immediate reward
    rho_c = get_rho(agent_i, c, r_picker, r_other)
    immediate = float(m[c]) * rho_c
    
    # Future value: actor goes to RAND
    v_rand_c, _, _ = calculate_stream_values(rho_c, gamma, alpha, beta, p_hit)
    
    # Future value: non-actors stay ON or OFF
    future_others = 0.0
    for j in range(n_agents):
        if j != c:
            rho_j = get_rho(agent_i, j, r_picker, r_other)
            v_rand_j, v_off_j, v_on_j = calculate_stream_values(rho_j, gamma, alpha, beta, p_hit)
            if m[j]:
                future_others += v_on_j
            else:
                future_others += v_off_j
    
    return immediate + gamma * (v_rand_c + future_others)

In [25]:
# Compute analytical values for ALL states and ALL agents
analytical_values: Dict[SimpleState, Dict[int, float]] = {}

for state in ALL_STATES:
    analytical_values[state] = {}
    for i in range(N_AGENTS):
        analytical_values[state][i] = analytical_value(
            state, i, R_PICKER, R_OTHER, GAMMA, ALPHA_PROB, BETA_PROB, P_HIT
        )

# Print sample
sample_state = ALL_STATES[0]
print(f"Sample state: m={sample_state.m}, c={sample_state.c}")
for i in range(N_AGENTS):
    print(f"  V^({i})(s) = {analytical_values[sample_state][i]:.6f}")
print(f"RAM after analytical: {get_ram_mb():.1f} MB")

Sample state: m=(False, False, False, False), c=0
  V^(0)(s) = 11.272923
  V^(1)(s) = 12.064006
  V^(2)(s) = 12.064006
  V^(3)(s) = 12.064006
RAM after analytical: 601.4 MB


In [26]:
# Environment dynamics

def get_reward(state: SimpleState, agent_i: int, r_picker: float, r_other: float) -> float:
    """Get reward for agent i given state."""
    if state.m[state.c]:  # actor is ON apple
        return r_picker if agent_i == state.c else r_other
    return 0.0


def step(state: SimpleState, p_hit: float, n_agents: int) -> SimpleState:
    """Transition to next state."""
    m_list = list(state.m)
    m_list[state.c] = bool(np.random.random() < p_hit)
    new_c = int(np.random.randint(0, n_agents))
    return SimpleState(m=tuple(m_list), c=new_c)

In [27]:
# Value Function Base Class

class ValueFunction:
    """Base class for value function approximation."""
    
    def get_value(self, state: SimpleState, agent_i: int) -> float:
        raise NotImplementedError
    
    def update(self, state: SimpleState, agent_i: int, reward: float, 
               next_state: SimpleState, gamma: float) -> float:
        """TD0 update. Returns TD error."""
        raise NotImplementedError

In [28]:
class TabularValue(ValueFunction):
    """Tabular value function: dict[(state, agent)] -> float"""
    
    def __init__(self, n_agents: int, alpha: float):
        self.n_agents = n_agents
        self.alpha = alpha
        self.V: Dict[Tuple[SimpleState, int], float] = {}
    
    def get_value(self, state: SimpleState, agent_i: int) -> float:
        key = (state, agent_i)
        return self.V.get(key, 0.0)
    
    def update(self, state: SimpleState, agent_i: int, reward: float,
               next_state: SimpleState, gamma: float) -> float:
        key = (state, agent_i)
        v_curr = self.V.get(key, 0.0)
        v_next = self.get_value(next_state, agent_i)
        td_target = reward + gamma * v_next
        td_error = td_target - v_curr
        self.V[key] = v_curr + self.alpha * td_error
        return abs(td_error)

In [29]:
class DQNOneHotValue(ValueFunction):
    """
    DQN on one-hot encoded state. 
    Supports toggling Replay Buffer and Target Network for isolation experiments.
    """
    
    def __init__(self, n_agents: int, hidden_dim: int, num_layers: int, lr: float, device: torch.device,
                 use_buffer: bool, use_target: bool):
        self.n_agents = n_agents
        self.device = device
        self.use_buffer = use_buffer
        self.use_target = use_target
        
        self.train_steps = 0
        input_dim = 2 * n_agents
        
        # Storage for independent components per agent
        self.networks = []
        self.target_networks = []  # New
        self.optimizers = []
        self.schedulers = []
        self.buffers = []          # New
        
        for i in range(n_agents):
            # 1. Build Policy Net
            layers = []
            in_d = input_dim
            for _ in range(num_layers):
                layers.append(nn.Linear(in_d, hidden_dim))
                layers.append(nn.ReLU())
                in_d = hidden_dim
            layers.append(nn.Linear(in_d, 1))
            net = nn.Sequential(*layers).to(device)
            
            # 2. Safe Initialization (From your provided code)
            with torch.no_grad():
                # Kaiming for hidden
                for m in net.modules():
                    if isinstance(m, nn.Linear):
                        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                        if m.bias is not None: nn.init.constant_(m.bias, 0.0)
                # Small Uniform for Output
                last_layer = layers[-1]
                nn.init.uniform_(last_layer.weight, -0.01, 0.01)
                if last_layer.bias is not None: nn.init.constant_(last_layer.bias, 0.0)
            
            self.networks.append(net)
            self.optimizers.append(torch.optim.Adam(net.parameters(), lr=lr))
            
            # 3. Build Target Net (if enabled)
            if self.use_target:
                target_net = copy.deepcopy(net)
                target_net.eval()
                self.target_networks.append(target_net)
            else:
                self.target_networks.append(None) # Placeholder
                
            # 4. Build Replay Buffer (if enabled)
            if self.use_buffer:
                self.buffers.append(ReplayBuffer(BUFFER_SIZE))
            else:
                self.buffers.append(None)
            
            self.schedulers.append(
                torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizers[-1], T_max=TRAIN_STEPS, eta_min=LR_MIN)
            )

    def _encode(self, state: SimpleState) -> torch.Tensor:
        x = state.to_onehot(self.n_agents)
        return torch.tensor(x, dtype=torch.float32, device=self.device).unsqueeze(0)
    
    def get_value(self, state: SimpleState, agent_i: int) -> float:
        x = self._encode(state)
        net = self.networks[agent_i]
        with torch.no_grad():
            return net(x).item()
    
    def update(self, state: SimpleState, agent_i: int, reward: float,
               next_state: SimpleState, gamma: float) -> float:
        
        # --- 1. Prepare Data ---
        s_tensor = self._encode(state)
        ns_tensor = self._encode(next_state)
        r_tensor = torch.tensor([[reward]], dtype=torch.float32, device=self.device)
        
        buffer = self.buffers[agent_i]
        
        # Add to buffer if enabled
        if self.use_buffer:
            buffer.push(s_tensor, r_tensor, ns_tensor)
            
            # If not enough data, skip update (return 0 error)
            if len(buffer) < BATCH_SIZE:
                return 0.0
            
            # Sample Batch
            b_state, b_reward, b_next_state = buffer.sample(BATCH_SIZE)
        else:
            # Pure Online: Batch is just the current transition
            b_state, b_reward, b_next_state = s_tensor, r_tensor, ns_tensor

        # --- 2. Compute Loss ---
        net = self.networks[agent_i]
        target_net = self.target_networks[agent_i]
        opt = self.optimizers[agent_i]
        
        # Current Q
        v_curr = net(b_state)
        
        # Target Q
        with torch.no_grad():
            if self.use_target:
                v_next = target_net(b_next_state)
            else:
                v_next = net(b_next_state) # Bootstrap off self
            
            td_target = b_reward + gamma * v_next
            
        loss = nn.MSELoss()(v_curr, td_target)
        
        # --- 3. Optimize ---
        opt.zero_grad()
        loss.backward()
        # Optional: Clip Grad
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        self.schedulers[agent_i].step()
        
        # --- 4. Update Target Net (Global Step Counter) ---
        # We only increment a counter once per step (managed externally) or checking here.
        # Since update is called N times per step, we just do a modulo check.
        # Note: In this loop structure, 'update' is called often. 
        # A simple robust way: 
        if self.use_target:
            # We use a randomized check or internal counter to avoid strictly coupling to global step
            # Or just use the scheduler's last epoch which tracks steps
            if self.schedulers[agent_i].last_epoch % TARGET_UPDATE_FREQ == 0:
                target_net.load_state_dict(net.state_dict())
                
        return loss.item() # Return MSE loss as error metric

In [30]:
class DQN5ChValue(ValueFunction):
    """
    DQN on 5-channel position encoding.
    Supports toggling Replay Buffer and Target Network for isolation experiments.
    """
    
    def __init__(self, n_agents: int, height: int, width: int, num_apples: int,
                 hidden_dim: int, num_layers: int, lr: float, device: torch.device,
                 use_buffer: bool, use_target: bool):
        self.n_agents = n_agents
        self.height = height
        self.width = width
        self.device = device
        self.use_buffer = use_buffer
        self.use_target = use_target
        
        self.train_steps = 0
        
        # Fixed apple grid setup
        self.apple_grid = np.zeros((height, width), dtype=np.float32)
        apple_indices = np.random.choice(height * width, size=num_apples, replace=False)
        for idx in apple_indices:
            r, c = divmod(idx, width)
            self.apple_grid[r, c] = 1.0
        
        self.on_positions = list(zip(*np.where(self.apple_grid > 0)))
        self.off_positions = list(zip(*np.where(self.apple_grid == 0)))
        
        input_dim = 5 * height * width
        
        # Storage for independent components per agent
        self.networks = []
        self.target_networks = []
        self.optimizers = []
        self.schedulers = []
        self.buffers = []
        
        for i in range(n_agents):
            # 1. Build Policy Net
            layers = []
            in_d = input_dim
            for _ in range(num_layers):
                layers.append(nn.Linear(in_d, hidden_dim))
                layers.append(nn.ReLU())
                in_d = hidden_dim
            layers.append(nn.Linear(in_d, 1))
            net = nn.Sequential(*layers).to(device)
            
            # 2. Safe Initialization
            with torch.no_grad():
                for m in net.modules():
                    if isinstance(m, nn.Linear):
                        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                        if m.bias is not None: nn.init.constant_(m.bias, 0.0)
                last_layer = layers[-1]
                nn.init.uniform_(last_layer.weight, -0.01, 0.01)
                if last_layer.bias is not None: nn.init.constant_(last_layer.bias, 0.0)
            
            self.networks.append(net)
            self.optimizers.append(torch.optim.Adam(net.parameters(), lr=lr))
            
            # 3. Build Target Net (if enabled)
            if self.use_target:
                target_net = copy.deepcopy(net)
                target_net.eval()
                self.target_networks.append(target_net)
            else:
                self.target_networks.append(None)
                
            # 4. Build Replay Buffer (if enabled)
            if self.use_buffer:
                self.buffers.append(ReplayBuffer(BUFFER_SIZE))
            else:
                self.buffers.append(None)
            
            self.schedulers.append(
                torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizers[-1], T_max=TRAIN_STEPS, eta_min=LR_MIN)
            )

    def _generate_positions(self, state: SimpleState) -> List[Tuple[int, int]]:
        positions = []
        for i, is_on in enumerate(state.m):
            if is_on:
                pos = self.on_positions[np.random.randint(len(self.on_positions))]
            else:
                pos = self.off_positions[np.random.randint(len(self.off_positions))]
            positions.append(pos)
        return positions
    
    def _encode(self, state: SimpleState, agent_i: int, positions: List[Tuple[int, int]]) -> torch.Tensor:
        H, W = self.height, self.width
        
        c_apples = self.apple_grid.copy()
        c_others = np.zeros((H, W), dtype=np.float32)
        c_self = np.zeros((H, W), dtype=np.float32)
        c_self_act = np.zeros((H, W), dtype=np.float32)
        c_other_act = np.zeros((H, W), dtype=np.float32)
        
        for k, pos in enumerate(positions):
            r, c = pos
            if k == agent_i:
                c_self[r, c] = 1.0
            else:
                c_others[r, c] += 1.0
        
        actor_pos = positions[state.c]
        if state.c == agent_i:
            c_self_act[actor_pos[0], actor_pos[1]] = 1.0
        else:
            c_other_act[actor_pos[0], actor_pos[1]] = 1.0
        
        x = np.stack([c_apples, c_others, c_self, c_self_act, c_other_act]).flatten()
        return torch.tensor(x, dtype=torch.float32, device=self.device).unsqueeze(0)
    
    def get_value(self, state: SimpleState, agent_i: int) -> float:
        positions = self._generate_positions(state)
        x = self._encode(state, agent_i, positions)
        net = self.networks[agent_i]
        with torch.no_grad():
            return net(x).item()
    
    def update(self, state: SimpleState, agent_i: int, reward: float,
               next_state: SimpleState, gamma: float) -> float:
        
        # --- 1. Generate & Encode ---
        # Note: We generate positions ONCE per update so the transition (s->s') is consistent
        current_positions = self._generate_positions(state)
        
        # Determine next positions consistent with transition logic
        next_positions = current_positions.copy()
        if next_state.m[state.c]: # Actor ended up ON an apple
            next_positions[state.c] = self.on_positions[np.random.randint(len(self.on_positions))]
        else: # Actor ended up OFF
            next_positions[state.c] = self.off_positions[np.random.randint(len(self.off_positions))]
            
        s_tensor = self._encode(state, agent_i, current_positions)
        ns_tensor = self._encode(next_state, agent_i, next_positions)
        r_tensor = torch.tensor([[reward]], dtype=torch.float32, device=self.device)
        
        buffer = self.buffers[agent_i]
        
        if self.use_buffer:
            buffer.push(s_tensor, r_tensor, ns_tensor)
            if len(buffer) < BATCH_SIZE:
                return 0.0
            b_state, b_reward, b_next_state = buffer.sample(BATCH_SIZE)
        else:
            b_state, b_reward, b_next_state = s_tensor, r_tensor, ns_tensor

        # --- 2. Compute Loss ---
        net = self.networks[agent_i]
        target_net = self.target_networks[agent_i]
        opt = self.optimizers[agent_i]
        
        v_curr = net(b_state)
        
        with torch.no_grad():
            if self.use_target:
                v_next = target_net(b_next_state)
            else:
                v_next = net(b_next_state)
            
            td_target = b_reward + gamma * v_next
            
        loss = nn.MSELoss()(v_curr, td_target)
        
        # --- 3. Optimize ---
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        self.schedulers[agent_i].step()
        
        # --- 4. Update Target Net ---
        if self.use_target:
            if self.schedulers[agent_i].last_epoch % TARGET_UPDATE_FREQ == 0:
                target_net.load_state_dict(net.state_dict())
                
        return loss.item()

In [31]:
# Initialize value function based on METHOD

if METHOD == "tabular":
    value_fn = TabularValue(N_AGENTS, ALPHA)
    print("Using Tabular TD0")
elif METHOD == "dqn_onehot":
    value_fn = DQNOneHotValue(
        N_AGENTS, HIDDEN_DIM, NUM_LAYERS, LR, DEVICE,
        use_buffer=USE_REPLAY_BUFFER,
        use_target=USE_TARGET_NET
    )
    print(f"Using DQN One-Hot (hidden={HIDDEN_DIM}, layers={NUM_LAYERS})")
elif METHOD == "dqn_5ch":
    value_fn = DQN5ChValue(
    N_AGENTS, HEIGHT, WIDTH, NUM_APPLES, HIDDEN_DIM, NUM_LAYERS, LR, DEVICE,
    use_buffer=USE_REPLAY_BUFFER,
    use_target=USE_TARGET_NET
)
    print(f"Using DQN 5-Channel (hidden={HIDDEN_DIM}, layers={NUM_LAYERS})")
else:
    raise ValueError(f"Unknown METHOD: {METHOD}")

print(f"RAM after model init: {get_ram_mb():.1f} MB")

Using DQN One-Hot (hidden=64, layers=2)
RAM after model init: 705.9 MB


In [32]:
def evaluate_all_states(value_fn: ValueFunction, all_states: List[SimpleState], 
                        analytical_values: Dict[SimpleState, Dict[int, float]], 
                        n_agents: int) -> Dict[str, Dict[str, float]]:
    """Evaluate value function on all states with robust metrics."""
    
    # 1. Collect all predictions and true values
    # Structure: agent_id -> {'preds': [], 'trues': []}
    agent_data = {i: {'preds': [], 'trues': []} for i in range(n_agents)}
    
    # Global lists for 'overall' stats
    all_preds_global = []
    all_trues_global = []

    for state in all_states:
        for i in range(n_agents):
            pred = value_fn.get_value(state, i)
            true = analytical_values[state][i]
            
            # Store per-agent
            agent_data[i]['preds'].append(pred)
            agent_data[i]['trues'].append(true)
            
            # Store global
            all_preds_global.append(pred)
            all_trues_global.append(true)

    # 2. Helper function to calculate statistics safely
    def calc_stats(preds_list: List[float], trues_list: List[float]) -> Dict[str, float]:
        p = np.array(preds_list, dtype=np.float64)
        t = np.array(trues_list, dtype=np.float64)
        
        errors = p - t
        abs_errors = np.abs(errors)
        
        # Safe MAPE: Only calculate percentage where True Value is not near zero
        # If true value is 0, percentage error is undefined. We exclude these points.
        nonzero_mask = np.abs(t) > 1e-4
        if np.sum(nonzero_mask) > 0:
            mape = np.mean(np.abs(errors[nonzero_mask] / t[nonzero_mask])) * 100
        else:
            mape = 0.0 # or np.nan, but 0.0 is safer for plotting if all values are 0
            
        # Safe Percent Bias (Drift)
        mean_t = np.mean(t)
        if abs(mean_t) > 1e-4:
            pct_bias = (np.mean(p) - mean_t) / mean_t * 100
        else:
            pct_bias = 0.0

        return {
            'mean_pred': float(np.mean(p)),
            'mean_true': float(np.mean(t)),
            'std_pred': float(np.std(p)),
            'std_true': float(np.std(t)),
            'mean_error': float(np.mean(abs_errors)),      # MAE
            'rmse': float(np.sqrt(np.mean(errors**2))),    # RMSE
            'mape': float(mape),
            'mean_bias': float(np.mean(errors)),
            'max_error': float(np.max(abs_errors)),
            'percent_bias': float(pct_bias),
            'std_bias': float(np.std(errors)),
        }

    # 3. Build Summary Dictionary
    summary = {}
    
    # Overall
    summary['overall'] = calc_stats(all_preds_global, all_trues_global)
    
    # Per-Agent (using string keys now)
    for i in range(n_agents):
        summary[f'agent_{i}'] = calc_stats(agent_data[i]['preds'], agent_data[i]['trues'])
    
    return summary

In [33]:
# Evaluate before training
print("=== Before Training ===")
eval_before = evaluate_all_states(value_fn, ALL_STATES, analytical_values, N_AGENTS)
print(f"Overall: Mean Error = {eval_before['overall']['mean_error']:.4f}, Mean Bias = {eval_before['overall']['mean_bias']:.4f}")
for i in range(N_AGENTS):
    print(f"  Agent {i}: Mean Error = {eval_before[f'agent_{i}']['mean_error']:.4f}, Mean Bias = {eval_before[f'agent_{i}']['mean_bias']:.4f}")

=== Before Training ===
Overall: Mean Error = 12.3491, Mean Bias = -12.3491
  Agent 0: Mean Error = 12.3261, Mean Bias = -12.3261
  Agent 1: Mean Error = 12.3262, Mean Bias = -12.3262
  Agent 2: Mean Error = 12.3642, Mean Bias = -12.3642
  Agent 3: Mean Error = 12.3799, Mean Bias = -12.3799


In [ ]:
# --- Setup Output Path & Static Data ---
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Generate filename early
if METHOD == "tabular":
    filename = f"tabular_td0_history_{METHOD}_p{PICKER_PENALTY}_s{SEED}_pid{os.getpid()}.csv"
else:
    filename = f"td0_history_{METHOD}_buf{USE_REPLAY_BUFFER}_tgt{USE_TARGET_NET}_p{PICKER_PENALTY}_s{SEED}_pid{os.getpid()}.csv"
output_file = output_dir / filename

# These columns don't change, so we prepare them once
static_params = {
    'method': METHOD,
    'picker_penalty': PICKER_PENALTY,
    'seed': SEED,
    'n_agents': N_AGENTS,
    'learning_rate_initial': ALPHA if METHOD == 'tabular' else LR,
    'use_buffer': USE_REPLAY_BUFFER,
    'use_target': USE_TARGET_NET
}

def save_live_row(row_data, filepath, first_write=False):
    """Helper to append a single row to CSV immediately."""
    # Combine dynamic stats with static config
    full_row = {**row_data, **static_params}
    
    # Create a 1-row DataFrame
    df_row = pd.DataFrame([full_row])
    
    # If first write, 'w'rite new file with headers. 
    # Otherwise 'a'ppend without headers.
    mode = 'w' if first_write else 'a'
    header = first_write
    
    df_row.to_csv(filepath, mode=mode, header=header, index=False)

print(f"Logging live to: {output_file}")

# --- Initialization ---
m_init = tuple(bool(x) for x in np.random.random(N_AGENTS) < P_HIT)
c_init = int(np.random.randint(0, N_AGENTS))
state = SimpleState(m=m_init, c=c_init)

td_errors_buffer = deque(maxlen=EVAL_FREQ) 

# --- Initial Evaluation (Step 0) ---
eval_start = evaluate_all_states(value_fn, ALL_STATES, analytical_values, N_AGENTS)

row_0 = {
    'step': 0,
    'td_error': 0.0,
    'lr': ALPHA if METHOD == 'tabular' else LR,
    'ram_mb': get_ram_mb()
}

# Flatten stats
for k, v in eval_start['overall'].items():
    row_0[f'overall_{k}'] = v
for i in range(N_AGENTS):
    agent_key = f"agent_{i}"
    for k, v in eval_start[agent_key].items():
        row_0[f'{agent_key}_{k}'] = v

# SAVE STEP 0 TO DISK
save_live_row(row_0, output_file, first_write=True)

print(f"Step {0:>6}: "
      f"MAE={eval_start['overall']['mean_error']:.4f} | "
      f"Pred={eval_start['overall']['mean_pred']:.2f}")


# --- Training Loop ---
print(f"\n=== Training for {TRAIN_STEPS} steps ===")

for t in range(TRAIN_STEPS):
    next_state = step(state, P_HIT, N_AGENTS)
    
    # 1. Update (TD0)
    current_step_td_errors = []
    for i in range(N_AGENTS):
        reward = get_reward(state, i, R_PICKER, R_OTHER)
        td_err = value_fn.update(state, i, reward, next_state, GAMMA)
        current_step_td_errors.append(td_err)
    
    td_errors_buffer.append(np.mean(current_step_td_errors))
    state = next_state
    
    # 2. Evaluation & Live Logging
    if (t + 1) % EVAL_FREQ == 0:
        # Run full evaluation
        eval_result = evaluate_all_states(value_fn, ALL_STATES, analytical_values, N_AGENTS)
        
        # Get dynamic params
        current_lr = ALPHA
        if isinstance(value_fn, (DQNOneHotValue, DQN5ChValue)):
            current_lr = value_fn.optimizers[0].param_groups[0]['lr']
            
        if td_errors_buffer:
            avg_td_err = np.mean(td_errors_buffer)
        else:
            avg_td_err = 0.0
        
        # Construct Row
        row = {
            'step': t + 1,
            'td_error': avg_td_err,
            'lr': current_lr,
            'ram_mb': get_ram_mb()
        }
        
        for k, v in eval_result['overall'].items():
            row[f'overall_{k}'] = v
        for i in range(N_AGENTS):
            agent_key = f"agent_{i}"
            for k, v in eval_result[agent_key].items():
                row[f'{agent_key}_{k}'] = v
        
        # SAVE LIVE
        save_live_row(row, output_file, first_write=False)
        
        # Print Status
        ov = eval_result['overall']
        print(f"Step {t+1:>6}: "
              f"MAE={ov['mean_error']:.4f} | "
              f"RMSE={ov['rmse']:.4f} | "
              f"MAPE={ov['mape']:6.2f}% | "
              f"Bias={ov['mean_bias']:+.4f}")

print("Training complete. History saved.")

Logging live to: outputs/td0_history_dqn_onehot_bufFalse_tgtFalse_p-1.0_s42_pid424634.csv
Step      0: MAE=12.3491 | Pred=0.00

=== Training for 1000 steps ===
Step     10: MAE=12.3046 | RMSE=12.3314 | MAPE= 99.62% | Bias=-12.3046
Step     20: MAE=12.2236 | RMSE=12.2499 | MAPE= 98.97% | Bias=-12.2236
Step     30: MAE=12.1199 | RMSE=12.1454 | MAPE= 98.14% | Bias=-12.1199
Step     40: MAE=11.9745 | RMSE=11.9993 | MAPE= 96.97% | Bias=-11.9745
Step     50: MAE=11.7613 | RMSE=11.7854 | MAPE= 95.26% | Bias=-11.7613
Step     60: MAE=11.5635 | RMSE=11.5894 | MAPE= 93.68% | Bias=-11.5635
Step     70: MAE=11.3169 | RMSE=11.3489 | MAPE= 91.70% | Bias=-11.3169
Step     80: MAE=11.0957 | RMSE=11.1342 | MAPE= 89.93% | Bias=-11.0957
Step     90: MAE=10.9023 | RMSE=10.9353 | MAPE= 88.38% | Bias=-10.9023
Step    100: MAE=10.6559 | RMSE=10.6789 | MAPE= 86.42% | Bias=-10.6559
Step    110: MAE=10.4862 | RMSE=10.5037 | MAPE= 85.07% | Bias=-10.4862
Step    120: MAE=10.2561 | RMSE=10.2758 | MAPE= 83.23% | Bi

In [35]:
# Final evaluation
print("\n=== Final Results ===")
eval_final = evaluate_all_states(value_fn, ALL_STATES, analytical_values, N_AGENTS)

print(f"\n{'Agent':<8} {'Mean Error':<12} {'Max Error':<12} {'Mean Bias':<12} {'Std Bias':<12}")
print("-" * 56)
for i in range(N_AGENTS):
    r = eval_final[f'agent_{i}']
    print(f"{i:<8} {r['mean_error']:<12.4f} {r['max_error']:<12.4f} {r['mean_bias']:<+12.4f} {r['std_bias']:<12.4f}")
print("-" * 56)
r = eval_final['overall']
print(f"{'Overall':<8} {r['mean_error']:<12.4f} {r['max_error']:<12.4f} {r['mean_bias']:<+12.4f} {r['std_bias']:<12.4f}")
print(f"\nFinal RAM: {get_ram_mb():.1f} MB")


=== Final Results ===

Agent    Mean Error   Max Error    Mean Bias    Std Bias    
--------------------------------------------------------
0        8.1420       9.4250       -8.1420      0.6367      
1        6.4790       8.2166       -6.4790      0.7195      
2        6.7377       9.0993       -6.7377      0.9172      
3        8.3486       9.4105       -8.3486      0.3852      
--------------------------------------------------------
Overall  7.4268       9.4250       -7.4268      1.0779      

Final RAM: 710.0 MB


In [36]:

print("\n=== All State Values (Agent 0) ===")
print(f"{'State (m, c)':<30} {'Predicted':<12} {'Analytical':<12} {'Error':<12}")
print("-" * 66)
for state in ALL_STATES[:20]:
    pred = value_fn.get_value(state, 0)
    true = analytical_values[state][0]
    err = pred - true
    print(f"{str((state.m, state.c)):<30} {pred:<12.4f} {true:<12.4f} {err:<+12.4f}")
print("... (showing first 20 states)")


=== All State Values (Agent 0) ===
State (m, c)                   Predicted    Analytical   Error       
------------------------------------------------------------------
((False, False, False, False), 0) 2.3854       11.2729      -8.8875     
((False, False, False, False), 1) 2.7750       12.0640      -9.2890     
((False, False, False, False), 2) 2.9282       12.0640      -9.1358     
((False, False, False, False), 3) 2.6390       12.0640      -9.4250     
((False, False, False, True), 0) 3.5388       11.9137      -8.3749     
((False, False, False, True), 1) 3.8337       12.7048      -8.8711     
((False, False, False, True), 2) 4.0300       12.7048      -8.6748     
((False, False, False, True), 3) 3.6789       12.7307      -9.0517     
((False, False, True, False), 0) 3.2150       11.9137      -8.6987     
((False, False, True, False), 1) 3.4297       12.7048      -9.2751     
((False, False, True, False), 2) 3.7308       12.7307      -8.9999     
((False, False, True, False), 3